<a href="https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule (plain words): Pages that sit at a given search position should get roughly the click-through rate that other pages at the same position get. A page whose CTR is well below the typical CTR for its position tier, and which has enough impressions to matter, is a candidate for a title and meta description review. I score each page by its estimated missed clicks: the clicks it would have received at its tier’s median CTR, minus the clicks it actually received. Pages are ranked by this score, highest first.

Reason code (one): low_ctr_for_position applies when a page has at least 200 impressions in March and its CTR is below its position tier’s median CTR.

Action label: “review title and meta description.”

Why position tiers: CTR naturally falls as position gets worse, so comparing a position-2 page to a position-12 page would be unfair. I compare each page only to pages in the same tier.

What this rule does not claim: it ranks pages to review first. It does not claim that a page’s title is the cause of low CTR, or that editing it will increase clicks.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

# One row per page per client for March, only pages with search data and enough volume
MIN_IMPRESSIONS = 200   # policy choice: below this, CTR is too noisy to trust

page = con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions,
       SUM(gsc_clicks) AS clicks,
       SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position
FROM march
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= {MIN_IMPRESSIONS}
""").df()
page["ctr"] = page["clicks"] / page["impressions"]
print("Pages in slice:", len(page))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages in slice: 84833


In [8]:
import numpy as np

# Position tiers (policy choice: simple, explainable cutoffs)
bins   = [0, 3, 10, 20, np.inf]
labels = ["pos 1-3", "pos 4-10", "pos 11-20", "pos 21+"]
page["position_tier"] = pd.cut(page["avg_position"], bins=bins, labels=labels, right=True)

# Expected CTR = median CTR of pages in the same tier
page["expected_ctr"] = page.groupby("position_tier", observed=True)["ctr"].transform("median")

# Score = estimated missed clicks (never negative)
page["missed_clicks"] = (page["expected_ctr"] * page["impressions"] - page["clicks"]).clip(lower=0)

# One reason code and one action label
page["reason_code"] = np.where(page["ctr"] < page["expected_ctr"], "low_ctr_for_position", "none")
page["action"] = np.where(page["reason_code"] == "low_ctr_for_position",
                          "review title and meta description", "monitor")

print(page.groupby("position_tier", observed=True)
          .agg(n=("ctr", "size"), median_ctr=("ctr", "median"))
          .round(4))
print("Pages flagged:", (page["reason_code"] == "low_ctr_for_position").sum())

                   n  median_ctr
position_tier                   
pos 1-3         9398      0.0022
pos 4-10       41864      0.0021
pos 11-20      16007      0.0014
pos 21+        17564      0.0003
Pages flagged: 42411


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, json

# Rank: highest missed clicks first; ties broken by impressions
queue = page.sort_values(["missed_clicks", "impressions"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

cols = ["rank", "client_hash_id", "content_hash_id", "impressions", "clicks", "ctr",
        "avg_position", "position_tier", "expected_ctr", "missed_clicks",
        "reason_code", "action"]
queue = queue[cols]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(queue))
print("Flagged (low_ctr_for_position):", (queue["reason_code"] == "low_ctr_for_position").sum())
queue.head(10)

Rows written: 84833
Flagged (low_ctr_for_position): 42411


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_tier,expected_ctr,missed_clicks,reason_code,action
0,1,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.000113,0.665877,pos 1-3,0.002245,452.756076,low_ctr_for_position,review title and meta description
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,0.000007,2.693038,pos 1-3,0.002245,301.981310,low_ctr_for_position,review title and meta description
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,0.000008,0.308426,pos 1-3,0.002245,277.495274,low_ctr_for_position,review title and meta description
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,0.000301,3.166132,pos 4-10,0.002079,254.336798,low_ctr_for_position,review title and meta description
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,0.000139,9.735658,pos 4-10,0.002079,208.667360,low_ctr_for_position,review title and meta description
5,6,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,0.000626,5.948459,pos 4-10,0.002079,192.661123,low_ctr_for_position,review title and meta description
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834.0,1.0,0.000012,0.116003,pos 1-3,0.002245,187.171451,low_ctr_for_position,review title and meta description
7,8,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,0.000045,7.831807,pos 4-10,0.002079,181.721414,low_ctr_for_position,review title and meta description
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,6.0,0.000072,7.208276,pos 4-10,0.002079,168.195426,low_ctr_for_position,review title and meta description
9,10,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.001420,2.468557,pos 1-3,0.002245,167.763673,low_ctr_for_position,review title and meta description


In [10]:
metrics = {
    "month": "2026-03",
    "min_impressions": MIN_IMPRESSIONS,
    "pages_in_slice": int(len(queue)),
    "pages_flagged": int((queue["reason_code"] == "low_ctr_for_position").sum()),
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

{'month': '2026-03', 'min_impressions': 200, 'pages_in_slice': 84833, 'pages_flagged': 42411}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top = queue.head(20).copy()

def confidence(r):
    if r["impressions"] >= 2000 and r["ctr"] < 0.5 * r["expected_ctr"]:
        return "high: lots of impressions and CTR is under half of the tier median"
    if r["impressions"] >= 500:
        return "medium: enough impressions, but a moderate gap"
    return "low: near the volume cutoff, CTR may be noisy"

def could_be_wrong(r):
    notes = []
    if r["impressions"] < 500:
        notes.append("low volume, the gap may be random noise")
    if r["avg_position"] % 1 > 0.8 or abs(r["avg_position"] - round(r["avg_position"])) < 0.05 and r["avg_position"] in (3, 10, 20):
        notes.append("position sits on a tier edge, so the comparison group may be unfair")
    notes.append("one month only, a seasonal dip could explain it")
    notes.append("the query mix may be informational, so low CTR may be normal for this intent")
    return "; ".join(notes[:3])

top["confidence"] = top.apply(confidence, axis=1)
top["what_would_make_it_wrong"] = top.apply(could_be_wrong, axis=1)

top[["rank", "content_hash_id", "impressions", "ctr", "expected_ctr", "avg_position",
     "position_tier", "reason_code", "action", "confidence", "what_would_make_it_wrong"]].round(4)

,rank,content_hash_id,impressions,ctr,expected_ctr,avg_position,position_tier,reason_code,action,confidence,what_would_make_it_wrong
0,1,content_44f34c0a90047651,212404.0,0.0001,0.0022,0.6659,pos 1-3,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
1,2,content_8e1334d6356668e3,134984.0,0.0000,0.0022,2.6930,pos 1-3,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
2,3,content_fec55986a1868d62,124075.0,0.0000,0.0022,0.3084,pos 1-3,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
3,4,content_34a70fea29d15f24,143019.0,0.0003,0.0021,3.1661,pos 4-10,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
4,5,content_f6116743b00afc2d,107584.0,0.0001,0.0021,9.7357,pos 4-10,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
5,6,content_7c6373141eae744a,132593.0,0.0006,0.0021,5.9485,pos 4-10,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"position sits on a tier edge, so the compariso..."
6,7,content_9c057b66c30a3abb,83834.0,0.0000,0.0022,0.1160,pos 1-3,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
7,8,content_cd3d932d4e1c8db0,89332.0,0.0000,0.0021,7.8318,pos 4-10,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"position sits on a tier edge, so the compariso..."
8,9,content_046fc480045b88f5,83788.0,0.0001,0.0021,7.2083,pos 4-10,low_ctr_for_position,review title and meta description,high: lots of impressions and CTR is under hal...,"one month only, a seasonal dip could explain i..."
9,10,content_8d7d99f109e19aa2,203497.0,0.0014,0.0022,2.4686,pos 1-3,low_ctr_for_position,review title and meta description,"medium: enough impressions, but a moderate gap","one month only, a seasonal dip could explain i..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: 3 of the top 20 look questionable (ranks 11, 13, and 20). Each has 70,000+ impressions but almost no clicks (3 to 42), including one at an average position near 1 with a CTR of about 0.06%. That is implausibly low for a true top result, so these may reflect a data artifact (for example, an average position blended across many different queries), a very low-intent query mix, or a tracking issue, not a real title problem. I would verify these before acting on them. The queue is also concentrated: one client accounts for 6 of the top 20, and all 20 sit in positions 1-10, so it reflects a few clients and the higher tiers more than the whole inventory.

Leakage check: the score uses only March 2026 values (impressions, clicks, average position, and CTR computed from them). The source dates run from 2026-03-01 to 2026-03-31, so no future window is used, and no product flags or scores appear in the queue (the banned-columns check printed an empty set).

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top = queue.head(20).copy()

# 1) Weak-pick candidates among the top 20
weak = top[
    (top["impressions"] < 500) |                                   # near the volume cutoff, noisy
    (top["clicks"] == 0) |                                         # zero clicks: possible tracking gap
    (top["avg_position"].sub(top["avg_position"].round()).abs() < 0.05)  # sits right on a whole-number position
]
print("Possible weak picks:", len(weak), "of 20")
print(weak[["rank", "impressions", "clicks", "ctr", "avg_position", "position_tier"]].round(4))

# Concentration check
print(top["client_hash_id"].value_counts().head())
print(top["position_tier"].value_counts())

# 2) Leakage check: every input the score used, and its source
used = ["impressions", "clicks", "avg_position", "ctr", "position_tier", "expected_ctr", "missed_clicks"]
print("Inputs to the score:", used)
print("Date range of source rows:", con.sql("SELECT MIN(report_date), MAX(report_date) FROM march").fetchone())
banned = {"health_score", "priority_score", "action_type", "trend_direction", "trend_pct"}
print("Banned columns present in queue:", banned & set(queue.columns))

Possible weak picks: 3 of 20
    rank  impressions  clicks     ctr  avg_position position_tier
10    11      82376.0    11.0  0.0001        8.0052      pos 4-10
12    13      71513.0     3.0  0.0000        6.9834      pos 4-10
19    20      70398.0    42.0  0.0006        0.9693       pos 1-3
client_hash_id
client_62f4a7e64f5e0096    6
client_73cda7b4e4f265ea    4
client_a80fca3f171ed1de    3
client_e547b89c05043229    3
client_23a62021009f63c4    2
Name: count, dtype: int64
position_tier
pos 4-10     12
pos 1-3       8
pos 11-20     0
pos 21+       0
Name: count, dtype: int64
Inputs to the score: ['impressions', 'clicks', 'avg_position', 'ctr', 'position_tier', 'expected_ctr', 'missed_clicks']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Date range of source rows: (datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))
Banned columns present in queue: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.